# P68 — STRIPS: un nuevo enfoque para aplicar la demostración de teoremas a la resolución de problemas

## 1. Título y paper

**Paper:** *STRIPS: A New Approach to the Application of Theorem Proving to Problem Solving*  
**Autoría:** Richard E. Fikes, Nils J. Nilsson  
**Año y venue:** 1971 · Artificial Intelligence, 2(3–4), 189–208  
**Nivel:** L2 · **Motor:** `strips`  
**Ficha completa:** [`P68_strips`](../../papers/foundational/P68_strips/README.md)

**Hito:** Da a la planificación su representación duradera —precondición, añadir, borrar— y con ella una respuesta práctica al problema del marco.

- [doi:10.1016/0004-3702(71)90010-5](https://doi.org/10.1016/0004-3702(71)90010-5)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Describir en lógica qué cambia y qué no al ejecutar una acción exigía escribir un axioma por cada literal que permanece igual. Es el problema del marco, y hacía inviable planificar con un demostrador de teoremas.
2. Ejecutar una implementación mínima de la propuesta: Describir cada operador con tres listas —precondiciones, literales que añade y literales que borra— y adoptar el supuesto de que todo lo no mencionado persiste.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P64
- P66
- McCarthy y Hayes (1969), el problema del marco


## 4. Intuición

Describir una acción parece fácil hasta que intentas escribir todo lo que **no** cambia. Si muevo un bloque, sigue habiendo una mesa, sigo teniendo dos manos y el color de las paredes no varía. STRIPS resuelve eso por convención: solo se declara lo que cambia, y lo demás persiste.


## 5. Concepto mínimo

```text
Operador = ⟨precondiciones, lista de añadir, lista de borrar⟩

    mover(C, A→mesa):
        pre : sobre(C,A), libre(C)
        add : sobre(C,mesa), libre(A)
        del : sobre(C,A)

Todo literal no mencionado en add ni en del SIGUE SIENDO CIERTO.
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('strips', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Cuántos literales que el operador no menciona siguen siendo ciertos?
2. ¿Resuelve el planificador lineal la meta si ataca «A sobre B» primero?
3. ¿Y si ataca «B sobre C» primero?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('strips', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('strips', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

Cuatro literales persisten sin que nadie los reafirme: ese es el problema del marco resuelto por convención. Y el planificador lineal consigue **1 de 2** submetas con un orden y **1 de 2** con el inverso: ningún orden funciona. Existe, sin embargo, un plan de 3 pasos que sí resuelve — pero intercala las submetas en vez de cerrarlas por turnos.


## 10. Comentario pedagógico

La anomalía de Sussman no es un defecto de implementación: es una propiedad del esquema. Cerrar una submeta antes de tocar la siguiente falla cuando las submetas interactúan, y eso motiva toda la planificación no lineal posterior. El mismo problema reaparece hoy en los agentes que descomponen una tarea en subtareas y las ejecutan en orden.


## 11. Error o anti-patrón deliberado

Anti-patrón: leer la anomalía como un fallo del dominio o del código.


In [ ]:
print('El mundo de bloques es trivial y el plan correcto tiene 3 pasos.')
print('Lo que falla es el ESQUEMA: cerrar una submeta antes de tocar la siguiente.')
print('Cambiar de dominio no lo arregla; cambiar de planificador, si.')

## 12. Corrección

El plan que sí funciona, y por qué:


In [ ]:
r = run_paper_lab('strips', seed=7)['result']
print('orden A->B primero :', r['plan_con_A_sobre_B_primero']['logra'])
print('orden B->C primero :', r['plan_con_B_sobre_C_primero']['logra'])
print('intercalado        :', r['plan_no_lineal_intercalado']['plan'])
print('resuelve           :', r['plan_no_lineal_intercalado']['todas'])

## 13. Desafío guiado

Aplica a mano el operador `mover(C,A→mesa)` sobre el estado inicial y lista qué literales cambian y cuáles persisten.


In [ ]:
r = run_paper_lab('strips', seed=3)['result']
show(r)

## 14. Desafío autónomo

Escribe en PDDL el mundo de bloques con estos operadores y resuelve la anomalía de Sussman con un planificador real. Después compara la longitud del plan con los 3 pasos del intercalado.


## 15. Evidencia de aprendizaje

Guarda la representación del operador con sus tres listas y tu explicación de por qué ningún orden lineal resuelve la meta.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P68_strips/README.md) · evaluación formal: [`assessments/papers/P68_strips.md`](../../assessments/papers/P68_strips.md)


## 16. Cierre

La planificación ya tiene representación. Pero el mundo real no da hechos ciertos: da indicios de fuerza variable, y hay que razonar con ellos.


## 17. Conexión con el siguiente hito

- P13
- P32

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
